In [ ]:
import cooler
import cooltools
from cooltools import insulation as compute_insulation
import bioframe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# plotting helpers
def pcolormesh_45deg(ax, matrix_c, start=0, resolution=1, *args, **kwargs):
    import itertools
    n = matrix_c.shape[0]
    start_pos_vector = [start + resolution * i for i in range(n + 1)]
    t = np.array([[1, 0.5], [-1, 0.5]])
    matrix_a = np.dot(np.array([(i[1], i[0])
                                for i in itertools.product(start_pos_vector[::-1],
                                                           start_pos_vector)]), t)
    x = matrix_a[:, 1].reshape(n + 1, n + 1)
    y = matrix_a[:, 0].reshape(n + 1, n + 1)
    im = ax.pcolormesh(x, y, np.flipud(matrix_c), *args, **kwargs)
    im.set_rasterized(True)
    return im

from matplotlib.ticker import EngFormatter
bp_formatter = EngFormatter("b")
def format_ticks(ax, x=True, y=True, rotate=True):
    if y: ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_bottom()
    if rotate: ax.tick_params(axis="x", rotation=45)

In [ ]:
sample_map = pd.read_csv(
    "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/scripts/replicates/lists/qc_passed_samples.csv"
)
print(sample_map.shape)
sample_map.head(2)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# plot_insulation_zoom
# ══════════════════════════════════════════════════════════════════════════════
BALANCED_DIR = "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/data/replicates/balanced"

def plot_insulation_zoom(
    sample_groups,
    chrom, pos,
    sample_map,
    balanced_dir=BALANCED_DIR,
    resolution=25_000,
    win_mult=5,
    flanks=None,
    color_map=None,
    ncols=4,
    figsize=None,
    save_path=None,
):
    """
    End-to-end insulation snippet plot for any set of sample groups.

    Parameters
    ----------
    sample_groups : dict
        Keys are display labels. Values are dicts of {column: value} filters
        applied to sample_map to find sample IDs. E.g.:
            {
                "Vehicle": {"treatment": "vehicle",  "experiment": "experiment-19"},
                "TSA 4h":  {"treatment": "TSA_4h",   "experiment": "experiment-19"},
            }
    chrom, pos : str, int
        Chromosome and position (bp) to centre on.
    sample_map : pd.DataFrame
        The full sample sheet (must have a "sample_id" column).
    balanced_dir : str
        Root directory containing per-sample cooler subdirectories.
    resolution : int   Cooler bin size in bp.
    win_mult : int     Insulation window = win_mult * resolution (default 5 -> 125 kb).
    flanks : list      Flank sizes in bp. Default: [25k,50k,100k,250k,400k,500k,1Mb].
    color_map : dict   {label: hex color}. Auto-assigned if None.
    ncols : int        Subplot grid columns.
    save_path : str    If given, saves the figure here.
    """
    if flanks is None:
        flanks = [25_000, 50_000, 100_000, 250_000, 400_000, 500_000, 1_000_000]

    default_colors = ["#1565C0", "#C62828", "#2E7D32", "#6A1B9A", "#E65100", "#00838F"]
    if color_map is None:
        color_map = {lbl: default_colors[i % len(default_colors)]
                     for i, lbl in enumerate(sample_groups)}

    win_bp  = win_mult * resolution
    win_col = f"log2_insulation_score_{win_bp}"

    # 1. find sample IDs
    def find_sids(filters):
        mask = pd.Series(True, index=sample_map.index)
        for col, val in filters.items():
            mask &= sample_map[col] == val
        return sample_map[mask]["sample_id"].tolist()

    # 2. load coolers
    def load_clrs(sids):
        clrs = []
        for sid in sids:
            path = f"{balanced_dir}/{sid}/{sid}_noMT_noY_noX.mcool::resolutions/{resolution}"
            try:
                clrs.append(cooler.Cooler(path))
            except Exception as e:
                print(f"  skipping {sid}: {e}")
        return clrs

    # 3. compute insulation
    def compute_ins(clrs):
        return [compute_insulation(c, [win_bp], verbose=False) for c in clrs]

    # 4. extract profile at a locus using bin-index arithmetic
    def extract_profiles(ins_list, chrom, pos, flank_bp):
        target_bin = pos // resolution
        profiles, x_kb_out = [], None
        for ins in ins_list:
            for c in [chrom, chrom.lstrip("chr"), "chr" + chrom.lstrip("chr")]:
                sub = ins[ins["chrom"] == c].copy()
                if not sub.empty:
                    break
            if sub.empty:
                profiles.append(None)
                continue
            sub["bin"] = sub["start"] // resolution
            sub = sub.set_index("bin")
            flank_bins = flank_bp // resolution
            idx = range(target_bin - flank_bins, target_bin + flank_bins + 1)
            vals = np.array([sub.loc[i, win_col] if i in sub.index else np.nan for i in idx])
            x_kb = np.array([(i - target_bin) * resolution / 1_000 for i in idx])
            if x_kb_out is None:
                x_kb_out = x_kb
            profiles.append(vals)
        return x_kb_out, profiles

    # ── run pipeline ──────────────────────────────────────────────────────────
    print("Loading samples and computing insulation...")
    group_ins = {}
    for label, filters in sample_groups.items():
        sids = find_sids(filters)
        print(f"  {label}: {len(sids)} samples found")
        clrs = load_clrs(sids)
        group_ins[label] = compute_ins(clrs)
        print(f"  {label}: insulation computed")

    # ── plot ──────────────────────────────────────────────────────────────────
    nrows = int(np.ceil(len(flanks) / ncols))
    if figsize is None:
        figsize = (ncols * 3, nrows * 2.8)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes_flat = np.array(axes).flatten()

    for ax, flank_bp in zip(axes_flat, flanks):
        for label, ins_list in group_ins.items():
            color = color_map[label]
            x_kb, profiles = extract_profiles(ins_list, chrom, pos, flank_bp)
            if x_kb is None:
                continue
            valid = [p for p in profiles if p is not None]
            for p in valid:
                ax.plot(x_kb, p, color=color, lw=0.6, alpha=0.3)
            if valid:
                ax.plot(x_kb, np.nanmean(valid, axis=0), color=color, lw=2.0,
                        label=f"{label} (n={len(valid)})" if ax is axes_flat[0] else None)

        ax.axvline(0, color="k", lw=0.8, ls="--", alpha=0.5)
        ax.axhline(0, color="grey", lw=0.4, ls=":")
        lbl = f"+-{flank_bp//1_000}kb" if flank_bp < 1_000_000 else "+-1Mb"
        ax.set_title(lbl, fontsize=9)
        ax.set_xlabel("Distance (kb)", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    axes_2d = np.array(axes).reshape(nrows, ncols)
    for ax in axes_2d[:, 0]:
        ax.set_ylabel("Insulation score (log2)", fontsize=8)
    for ax in axes_flat[len(flanks):]:
        ax.set_visible(False)

    fig.legend(frameon=False, fontsize=8, loc="upper right", bbox_to_anchor=(0.98, 0.98))
    plt.suptitle(f"{chrom}:{pos//1_000:,}kb", fontsize=10, y=1.02)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()

In [ ]:
# MainSix boundary file
MAINSIX_BED = (
    "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/outputs/global_calling/TAD/"
    "merged_mouse-primary-cerebellar-granule_treatment_vehicle_noXY/25kb/"
    "TAD_merged_mouse-primary-cerebellar-granule_treatment_vehicle_noXY_"
    "25kb_min75kb_max125kb_step25kb_thr001_fdr001_boundaries.bed"
)
boundaries_granule = pd.read_csv(
    MAINSIX_BED, sep="	", header=None,
    names=["chrom","start","end","name","score","strand"]
)
boundaries_granule["chrom"] = boundaries_granule["chrom"].astype(str)
print(f"{len(boundaries_granule):,} granule boundaries loaded")
boundaries_granule.head()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# browse_boundaries  +  load_isolated_boundaries
# ══════════════════════════════════════════════════════════════════════════════
TAD_DIR = (
    "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/"
    "outputs/global_calling/TAD"
)
BOUNDARY_FILES = {
    "astrocyte": (
        f"{TAD_DIR}/merged_human-primary-astrocytes_treatment_vehicle_noXY/25kb/"
        "TAD_merged_human-primary-astrocytes_treatment_vehicle_noXY_"
        "25kb_min75kb_max125kb_step25kb_thr001_fdr001_boundaries.bed"
    ),
    "hmc3": (
        f"{TAD_DIR}/merged_human_hmc3_treatment_vehicle_noXY/25kb/"
        "TAD_merged_human_hmc3_treatment_vehicle_noXY_"
        "25kb_min75kb_max125kb_step25kb_thr001_fdr001_boundaries.bed"
    ),
}
# which cell-type boundary file to use per experiment
EXP_CELLTYPE = {
    "experiment-08": "astrocyte",
    "experiment-12": "astrocyte",
    "experiment-13": "hmc3",
}

_DEFAULT_COLORS = ["#1565C0","#C62828","#2E7D32","#6A1B9A","#E65100","#00838F"]


def load_isolated_boundaries(bed_path, min_isolation=500_000):
    """Return boundaries >=min_isolation bp from any neighbour (both directions)."""
    df = pd.read_csv(
        bed_path, sep="\t", header=None,
        names=["chrom","start","end","name","score","strand"],
    )
    df["chrom"] = df["chrom"].astype(str)
    df["mid"] = (df["start"] + df["end"]) // 2
    df = df.sort_values(["chrom","mid"]).reset_index(drop=True)
    dist_prev = df.groupby("chrom")["mid"].diff().fillna(np.inf)
    dist_next = df.groupby("chrom")["mid"].diff(-1).abs().fillna(np.inf)
    out = df[(dist_prev >= min_isolation) & (dist_next >= min_isolation)].copy()
    fname = bed_path.split("/")[-1]
    print(f"{len(out)} isolated boundaries from {fname}")
    return out.reset_index(drop=True)


def _find_sids(filters, sample_map):
    mask = pd.Series(True, index=sample_map.index)
    for col, val in filters.items():
        mask &= sample_map[col] == val
    return sample_map[mask]["sample_id"].tolist()


def _load_clrs(sids, balanced_dir, resolution):
    clrs = []
    for sid in sids:
        path = f"{balanced_dir}/{sid}/{sid}_noMT_noY_noX.mcool::resolutions/{resolution}"
        try:
            clrs.append(cooler.Cooler(path))
        except Exception as e:
            print(f"  skipping {sid}: {e}")
    return clrs


def _extract_profile(ins_list, chrom, pos, win_col, flank_bp, resolution):
    target_bin = pos // resolution
    profiles, x_kb_out = [], None
    for ins in ins_list:
        sub = pd.DataFrame()
        for c in [chrom, chrom.lstrip("chr"), "chr" + chrom.lstrip("chr")]:
            sub = ins[ins["chrom"] == c].copy()
            if not sub.empty:
                break
        if sub.empty:
            profiles.append(None)
            continue
        sub["bin"] = sub["start"] // resolution
        sub = sub.set_index("bin")
        flank_bins = flank_bp // resolution
        idx = range(target_bin - flank_bins, target_bin + flank_bins + 1)
        vals = np.array([sub.loc[i, win_col] if i in sub.index else np.nan for i in idx])
        x_kb = np.array([(i - target_bin) * resolution / 1_000 for i in idx])
        if x_kb_out is None:
            x_kb_out = x_kb
        profiles.append(vals)
    return x_kb_out, profiles


def _load_group_ins(sample_groups, sample_map, balanced_dir, resolution, win_bp):
    group_ins = {}
    for label, filters in sample_groups.items():
        sids = _find_sids(filters, sample_map)
        print(f"  {label}: {len(sids)} samples found")
        clrs = _load_clrs(sids, balanced_dir, resolution)
        group_ins[label] = [
            compute_insulation(c, [win_bp], verbose=False) for c in clrs
        ]
        print(f"  {label}: insulation computed")
    return group_ins


def browse_boundaries(
    sample_groups,
    boundaries,
    sample_map,
    balanced_dir=BALANCED_DIR,
    resolution=25_000,
    win_mult=5,
    flank_bp=250_000,
    n_boundaries=12,
    ncols=4,
    color_map=None,
    figsize=None,
    save_path=None,
):
    """
    Grid plot: one subplot per boundary, insulation profiles for each group.

    Parameters
    ----------
    boundaries : pd.DataFrame
        Must have columns chrom, mid (bp).  Use load_isolated_boundaries().
    n_boundaries : int
        How many boundaries to plot (taken from the top of the df).
    flank_bp : int
        Half-window in bp for each subplot.
    """
    if color_map is None:
        color_map = {lbl: _DEFAULT_COLORS[i % len(_DEFAULT_COLORS)]
                     for i, lbl in enumerate(sample_groups)}

    win_bp  = win_mult * resolution
    win_col = f"log2_insulation_score_{win_bp}"

    print("Loading samples and computing insulation...")
    group_ins = _load_group_ins(sample_groups, sample_map, balanced_dir, resolution, win_bp)

    sub_bounds = boundaries.head(n_boundaries)
    nrows = int(np.ceil(len(sub_bounds) / ncols))
    if figsize is None:
        figsize = (ncols * 3.5, nrows * 2.8)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes_flat = np.array(axes).flatten()

    for ax, (_, row) in zip(axes_flat, sub_bounds.iterrows()):
        chrom, pos = str(row["chrom"]), int(row["mid"])
        for label, ins_list in group_ins.items():
            color = color_map[label]
            x_kb, profiles = _extract_profile(ins_list, chrom, pos, win_col, flank_bp, resolution)
            if x_kb is None:
                continue
            valid = [p for p in profiles if p is not None]
            for p in valid:
                ax.plot(x_kb, p, color=color, lw=0.5, alpha=0.25)
            if valid:
                ax.plot(x_kb, np.nanmean(valid, axis=0), color=color, lw=1.8,
                        label=label)
        ax.axvline(0, color="k", lw=0.8, ls="--", alpha=0.5)
        ax.axhline(0, color="grey", lw=0.4, ls=":")
        ax.set_title(f"{chrom}:{pos//1_000:,}kb", fontsize=8)
        ax.spines[["top","right"]].set_visible(False)

    axes_2d = np.array(axes).reshape(nrows, ncols)
    for ax in axes_2d[:, 0]:
        ax.set_ylabel("Ins. score (log2)", fontsize=8)
    for ax in axes_2d[-1]:
        ax.set_xlabel("Distance (kb)", fontsize=8)
    for ax in axes_flat[len(sub_bounds):]:
        ax.set_visible(False)

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, fontsize=7,
               loc="upper right", bbox_to_anchor=(0.98, 0.98))
    flank_lbl = f"+-{flank_bp//1_000}kb" if flank_bp < 1_000_000 else "+-1Mb"
    plt.suptitle(f"Isolated boundaries  ({flank_lbl})", fontsize=10, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


In [ ]:
# ── browse isolated boundaries per experiment ─────────────────────────────────
SAVE_DIR = (
    "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/outputs/"
)

samples1_filters = {
    "exp-12 Vehicle":    {"treatment": "vehicle",    "experiment": "experiment-12"},
    "exp-12 Loratadine": {"treatment": "Loratadine", "experiment": "experiment-12"},
    "exp-08 Vehicle":    {"treatment": "vehicle",    "experiment": "experiment-08"},
    "exp-08 H89":        {"treatment": "H89",        "experiment": "experiment-08"},
    "exp-13 Vehicle":    {"treatment": "vehicle",    "experiment": "experiment-13"},
    "exp-13 TC-E-5003":  {"treatment": "TC-E-5003",  "experiment": "experiment-13"},
}

isolated_boundaries = {
    ct: load_isolated_boundaries(path)
    for ct, path in BOUNDARY_FILES.items()
}

for exp, ct in EXP_CELLTYPE.items():
    short = exp.replace("experiment-", "exp-")
    groups_exp = {k: v for k, v in samples1_filters.items()
                  if k.startswith(short)}
    browse_boundaries(
        sample_groups=groups_exp,
        boundaries=isolated_boundaries[ct],
        sample_map=sample_map,
        n_boundaries=12,
        save_path=f"{SAVE_DIR}browse_boundaries_{exp}_{ct}.pdf",
    )


In [ ]:
# ── example: experiment-19 vehicle vs TSA 4h ─────────────────────────────────
plot_insulation_zoom(
    sample_groups={
        "Vehicle": {"treatment": "vehicle", "experiment": "experiment-19"},
        "TSA 4h":  {"treatment": "TSA_4h",  "experiment": "experiment-19"},
    },
    chrom="chr13", pos=28_950_000,
    sample_map=sample_map,
    save_path=(
        "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/outputs/"
        "insulation_snippet_chr13_28950kb_TSA4h_vs_vehicle.pdf"
    ),
)

In [ ]:
# ── samples1: multi-experiment comparison ─────────────────────────────────────
samples1 = {
    "experiment-12_vehicle":   {"treatment": "vehicle",    "experiment": "experiment-12"},
    "experiment-12_treatment": {"treatment": "Loratadine", "experiment": "experiment-12"},
    "experiment-08_vehicle":   {"treatment": "vehicle",    "experiment": "experiment-08"},
    "experiment-08_treatment": {"treatment": "H89",        "experiment": "experiment-08"},
    "experiment-13_vehicle":   {"treatment": "vehicle",    "experiment": "experiment-13"},
    "experiment-13_treatment": {"treatment": "TC-E-5003",  "experiment": "experiment-13"},
}

plot_insulation_zoom(
    sample_groups=samples1,
    chrom="chr13", pos=28_950_000,
    sample_map=sample_map,
    save_path=(
        "/Users/perera/oak/tttt/users/perera/research/plate_c/FINAL/outputs/"
        "insulation_snippet_chr13_28950kb_samples1.pdf"
    ),
)